# Robot de tri médical — entraînement des yeux (version YOLO, v2 : 3 jeux de données)

**Partie A** : détecteur YOLO (Ultralytics, licence **AGPL-3.0** — prévenir Hiba).
Il apprend sur **3 jeux Roboflow fusionnés** : le vôtre (Hospital Consumables) + 2 jeux publics
(Pharmaceutical & Biomedical Waste — domaine public ; Medical dataset de Vedant — CC BY 4.0, à citer).
**Partie B** : classifieur (dit *ce que c'est* sur l'objet découpé), images images.cv.

Avant de commencer :
1. **Exécution → Modifier le type d'exécution → GPU T4**
2. 🔑 **Secrets** (barre de gauche) : `ROBOFLOW_API_KEY`, accès au notebook activé.
3. Lancer les cellules **dans l'ordre**. Les résultats vont dans Google Drive (`MyDrive/modeles/`).

## A1. Carte graphique, paquets, Google Drive

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics roboflow onnx onnxslim
# Si Colab propose « Redémarrer la session » après l'installation : cliquer, puis relancer cette cellule.

from google.colab import drive
drive.mount("/content/drive")
SORTIE_DET = "/content/drive/MyDrive/modeles/yolo_medical"
SORTIE_CLS = "/content/drive/MyDrive/modeles/classifieur_medical"
import os
os.makedirs(SORTIE_DET, exist_ok=True)
os.makedirs(SORTIE_CLS, exist_ok=True)

## A2. Télécharger les 3 jeux et les fusionner (5 à 15 min)
Chaque jeu nomme ses classes à sa façon (« Syringe », « syringes », « -syringe- »…).
On les ramène à **une seule liste** ; les classes non médicales ou inconnues sont retirées
(leurs images restent, comme exemples « sans objet »). **Regardez le tableau de correspondance affiché.**

In [ ]:
from roboflow import Roboflow
from google.colab import userdata
import yaml, json, shutil, re
from pathlib import Path
from collections import Counter

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))

# (nom court, workspace, projet, version ou None = la plus récente)
JEUX = [
    ("nous",   "star-developer-6073",   "hospital-consumables-7zdur",      1),
    ("dechet", "medical-waste-disposal", "pharmaceutical-biomedical-waste", None),
    ("vedant", "vedant-tal6h",           "medical-dataset-aptyr",           None),
]

# ➕ PLUS D'IMAGES : collez ici des liens Roboflow Universe (un par ligne, entre guillemets).
#    Exemple : "https://universe.roboflow.com/nom-du-workspace/nom-du-projet"
#    Vérifiez la licence sur la page de chaque jeu.
LIENS_EN_PLUS = [
]
for i, lien in enumerate(LIENS_EN_PLUS):
    morceaux = [m for m in lien.split("universe.roboflow.com/")[-1].split("/") if m]
    if len(morceaux) >= 2:
        version = int(morceaux[3]) if len(morceaux) >= 4 and morceaux[2] == "dataset" and morceaux[3].isdigit() else None
        JEUX.append((f"plus{i + 1}", morceaux[0], morceaux[1], version))
    else:
        print("lien non compris, ignoré :", lien)

# La liste finale des classes (l'ordre donne les numéros)
CLASSES = ["syringe", "gloves", "box-of-gloves", "mask", "box-of-masks", "bandage", "gauze", "dressing-tray"]

def classe_unifiee(nom):
    n = re.sub(r"[\s_]+", "-", nom.strip().lower()).strip("-")
    if n in CLASSES:
        return n
    if "box" in n and "glove" in n: return "box-of-gloves"
    if "box" in n and "mask" in n:  return "box-of-masks"
    if "syringe" in n:              return "syringe"
    if "glove" in n:                return "gloves"
    if "mask" in n:                 return "mask"
    if "bandage" in n or "band-aid" in n or "bandaid" in n: return "bandage"
    if "gauze" in n:                return "gauze"
    if "dressing" in n:             return "dressing-tray"
    return None   # retirée

def derniere_version(projet):
    try:
        return max(int(str(v.version).split("/")[-1]) for v in projet.versions())
    except Exception:
        return None

FUSION_DET = Path("/content/fusion_det")
shutil.rmtree(FUSION_DET, ignore_errors=True)
DOSSIERS = {"train": "train", "valid": "valid", "test": "test"}
compte = Counter()

for court, ws, nom_projet, version in JEUX:
    print(f"\n=== {court} : {ws}/{nom_projet} ===")
    try:
        projet = rf.workspace(ws).project(nom_projet)
        version = version or derniere_version(projet)
        jeu = None
        for v in ([version] if version else range(10, 0, -1)):
            try:
                jeu = projet.version(v).download("yolov8", location=f"/content/brut_{court}")
                print("version", v)
                break
            except Exception as e:
                last = e
        if jeu is None:
            raise last
    except Exception as e:
        print(f"✗ {court} ignoré : {str(e)[:300]}")
        continue

    racine = Path(jeu.location)
    noms = yaml.safe_load((racine / "data.yaml").read_text())["names"]
    noms = list(noms.values()) if isinstance(noms, dict) else list(noms)
    table = {i: classe_unifiee(n) for i, n in enumerate(noms)}
    for i, n in enumerate(noms):
        print(f"   {n!r:35} → {table[i] or '(retirée)'}")

    for split in DOSSIERS:
        imgs = racine / split / "images"
        if not imgs.exists():
            continue
        (FUSION_DET / split / "images").mkdir(parents=True, exist_ok=True)
        (FUSION_DET / split / "labels").mkdir(parents=True, exist_ok=True)
        for img in imgs.iterdir():
            nouveau = f"{court}_{img.name}"
            shutil.copy(img, FUSION_DET / split / "images" / nouveau)
            etiq = racine / split / "labels" / (img.stem + ".txt")
            lignes = []
            if etiq.exists():
                for ligne in etiq.read_text().split("\n"):
                    p = ligne.split()
                    if len(p) < 5:
                        continue
                    cible = table.get(int(p[0]))
                    if cible is None:
                        continue
                    vals = list(map(float, p[1:]))
                    if len(vals) > 4:   # contour (polygone) → boîte
                        xs, ys = vals[0::2], vals[1::2]
                        vals = [(min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)]
                    lignes.append(f"{CLASSES.index(cible)} " + " ".join(f"{v:.6f}" for v in vals[:4]))
                    compte[(split, cible)] += 1
            (FUSION_DET / split / "labels" / (Path(nouveau).stem + ".txt")).write_text("\n".join(lignes))

DATA_YAML = FUSION_DET / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(FUSION_DET), "train": "train/images", "val": "valid/images",
    "test": "test/images" if (FUSION_DET / "test/images").exists() else "valid/images",
    "names": {i: c for i, c in enumerate(CLASSES)},
}, allow_unicode=True))
racine = FUSION_DET

print("\n=== BILAN ===")
for split in DOSSIERS:
    d = FUSION_DET / split / "images"
    if d.exists():
        print(f"{split:6} {len(list(d.iterdir())):5} images |", {c: compte[(split, c)] for c in CLASSES})

## A3. Entraîner YOLO (1 à 2 h — laisser l'onglet ouvert)
Les réglages sont dans le formulaire à droite de la cellule :
- **MODELE** : `yolo11n` (petit, rapide) · `yolo11s` (conseillé) · `yolo11m` (plus précis, plus lent).
- **FREEZE** : 0 = tout apprendre (conseillé avec beaucoup d'images) · 10 = geler le tronc (peu d'images, plus rapide).
- **NOM_ESSAI** : changez-le à chaque essai (`essai2`, `essai3`…) pour comparer sans écraser.

En cas de `CUDA out of memory` : BATCH = 8. **Si Colab coupe en route** : relancez A1, puis cochez REPRENDRE et relancez cette cellule.

In [ ]:
from ultralytics import YOLO

MODELE = "yolo11s"   # @param ["yolo11n", "yolo11s", "yolo11m"]
FREEZE = 0           # @param {type:"integer"}
EPOCHS = 100         # @param {type:"integer"}
BATCH = 16           # @param {type:"integer"}
NOM_ESSAI = "essai2" # @param {type:"string"}
REPRENDRE = False    # @param {type:"boolean"}

if REPRENDRE:
    YOLO(f"{SORTIE_DET}/{NOM_ESSAI}/weights/last.pt").train(resume=True)
else:
    YOLO(f"{MODELE}.pt").train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=640,
        batch=BATCH,
        freeze=FREEZE or None,
        patience=20,      # s'arrête tout seul si ça ne progresse plus pendant 20 passes
        cos_lr=True,      # apprend vite au début, affine à la fin
        flipud=0.5,       # caméra vue de dessus : une seringue à l'envers reste une seringue
        degrees=10,       # légères rotations
        project=SORTIE_DET,
        name=NOM_ESSAI,
        exist_ok=True,
    )

## A4. Scores et test sur des images de test

In [ ]:
import glob, random
from IPython.display import Image as Img, display

POIDS = f"{SORTIE_DET}/{NOM_ESSAI}/weights/best.pt"
meilleur = YOLO(POIDS)

m = meilleur.val(data=str(DATA_YAML), split="test")
print(f"\nmAP50 global : {m.box.map50:.3f}   |   mAP50-95 : {m.box.map:.3f}")
for i, c in enumerate(m.box.ap_class_index):
    print(f"  {meilleur.names[int(c)]:15s} mAP50 = {m.box.ap50[i]:.3f}")

images_test = random.sample(sorted(glob.glob(str(racine / "test/images/*"))), 4)
for r in meilleur.predict(images_test, conf=0.6, save=True, project=SORTIE_DET, name=f"tests_{NOM_ESSAI}", exist_ok=True):
    print(Path(r.path).name, "→", [(meilleur.names[int(c)], round(float(s), 2)) for c, s in zip(r.boxes.cls, r.boxes.conf)])
for f in sorted(glob.glob(f"{SORTIE_DET}/tests_{NOM_ESSAI}/*"))[:4]:
    display(Img(filename=f, width=600))

## A4 bis. Tester sur VOS images (depuis votre ordinateur)
Une fenêtre « Choisir des fichiers » apparaît : prenez une ou plusieurs photos (jpg, png, webp). Seuil 0,6 : le modèle ne parle que s'il est assez sûr.

In [ ]:
from google.colab import files
from PIL import Image as PILImage
import io

envoyes = files.upload()
for nom_fichier, contenu in envoyes.items():
    img = PILImage.open(io.BytesIO(contenu)).convert("RGB")
    r = meilleur.predict(img, conf=0.6)[0]
    trouves = [(meilleur.names[int(c)], round(float(s), 2)) for c, s in zip(r.boxes.cls, r.boxes.conf)]
    print(nom_fichier, "→", trouves if trouves else "rien trouvé")
    display(PILImage.fromarray(r.plot()[:, :, ::-1]))

## A5. Exporter le détecteur en ONNX (fichier qui tourne hors ligne sur le Mac / le robot)

In [ ]:
chemin_onnx = meilleur.export(format="onnx", imgsz=640, simplify=True)
json.dump(meilleur.names, open(f"{SORTIE_DET}/classes.json", "w"), ensure_ascii=False, indent=1)
print("ONNX :", chemin_onnx)
!ls -la {SORTIE_DET}/{NOM_ESSAI}/weights

## B1. images.cv : télécharger les catégories médicales (autonome)
Les zips vont dans Drive (`MyDrive/datasets/imagescv/`) : après une déconnexion, relancer cette cellule ne retélécharge rien.
Licence images.cv / ImageNet : **recherche seulement** — à montrer à Hiba.

In [ ]:
import requests, secrets, time, zipfile, shutil
from pathlib import Path

API = "https://api.images.cv"
ENTETES = {
    "Content-Type": "application/json", "Accept": "application/json",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36",
    "Origin": "https://images.cv", "Referer": "https://images.cv/",
}
CATEGORIES_CLS = ["syringe", "bandage", "band-aid", "scalpel", "clamp", "straight-scissor",
                  "curved-scissor", "pill-bottle", "stethoscope", "mask", "neck-brace", "medicine-chest"]
ZIPS = Path("/content/drive/MyDrive/datasets/imagescv")
FUSION = Path("/content/icv_fusion")
ZIPS.mkdir(parents=True, exist_ok=True)

def appel(chemin, corps):
    r = requests.post(API + chemin, json=corps, headers=ENTETES, timeout=60)
    if "application/json" not in r.headers.get("Content-Type", ""):
        raise RuntimeError(f"{chemin} : réponse non JSON ({r.status_code}) : {r.text[:200]!r}")
    d = r.json()
    if r.status_code != 200 or d.get("isError"):
        raise RuntimeError(f"{chemin} : erreur {r.status_code} : {d}")
    return d["data"]

def nombre_images(nom):
    return appel("/get_items_image_category", {"category": nom, "start_after": 0}).get("sumCounter") or 0

def telecharger(nom, dest):
    corps = {"output_size": "size_256", "color_type": "color", "images": [], "categories": [nom],
             "augmentations": [], "split_set": [0.7, 0.15, 0.15], "link_id": secrets.token_hex(20),
             "request_type": "CALL_FROM_SEARCH", "user_email": [], "user_id": "", "augmentation_percent": 30}
    lien_id = appel("/download_images", corps)["link_id"]
    fin = time.time() + 1200
    while time.time() < fin:
        url = appel("/get_link", {"link_id": lien_id}).get("public_url")
        if url:
            with requests.get(url, stream=True, timeout=120) as r, open(dest, "wb") as f:
                for morceau in r.iter_content(1 << 20):
                    f.write(morceau)
            with open(dest, "rb") as f:
                if f.read(2) != b"PK":
                    dest.unlink()
                    raise RuntimeError("le fichier reçu n'est pas un zip")
            return
        time.sleep(5)
    raise RuntimeError("trop long : le zip n'a pas été prêt en 20 min")

bilan = {}
for slug in CATEGORIES_CLS:
    zip_path = ZIPS / f"{slug}.zip"
    if not (zip_path.exists() and zip_path.stat().st_size > 10_000):
        nom = next((s for s in [slug, slug.replace("-", "_")] if nombre_images(s)), None)
        if not nom:
            print(f"✗ {slug} : introuvable")
            continue
        for essai in range(3):
            try:
                print(f"↓ {slug}…")
                telecharger(nom, zip_path)
                break
            except Exception as e:
                print(f"   essai {essai + 1} raté : {str(e)[:400]}")
                time.sleep(60 * (essai + 1))
        else:
            print(f"✗ {slug} : abandonné pour l'instant (relancer plus tard)")
            continue
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            parties = Path(m).parts
            if "data" in parties and not m.endswith("/"):
                i = parties.index("data")
                if len(parties) >= i + 4 and parties[i + 1] in ("train", "val", "test"):
                    dst = FUSION / parties[i + 1] / slug / parties[-1]
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    with z.open(m) as src, open(dst, "wb") as out:
                        shutil.copyfileobj(src, out)
    bilan[slug] = {s: len(list((FUSION / s / slug).glob("*"))) for s in ("train", "val", "test")}
    print(f"✓ {slug} : {bilan[slug]}")

print("\nBILAN :", bilan)

## B2. Regarder les images de chaque classe
ImageNet : `clamp` peut être une pince de menuisier, `mask` un masque de carnaval… Pour retirer une classe : `!rm -rf /content/icv_fusion/*/clamp`

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

classes = sorted(p for p in (FUSION / "train").iterdir() if any(p.iterdir()))
fig, axes = plt.subplots(len(classes), 6, figsize=(12, 2 * len(classes)))
for ligne, c in zip(axes, classes):
    fichiers = list(c.iterdir())
    for ax, p in zip(ligne, random.sample(fichiers, min(6, len(fichiers)))):
        ax.imshow(Image.open(p))
    for ax in ligne:
        ax.axis("off")
    ligne[0].set_title(c.name, loc="left", fontsize=9)
plt.tight_layout()
plt.show()

## B3. Entraîner le classifieur (MobileNetV3) et l'exporter en ONNX

In [ ]:
import json
import torch, torch.nn as nn
from torchvision import datasets, transforms, models

dev = "cuda" if torch.cuda.is_available() else "cpu"
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tf_train = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.6, 1)), transforms.RandomHorizontalFlip(),
                               transforms.ColorJitter(0.3, 0.3, 0.3), transforms.ToTensor(), norm])
tf_eval = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), norm])
tr = datasets.ImageFolder(FUSION / "train", tf_train)
va = datasets.ImageFolder(FUSION / "val", tf_eval)
dl_tr = torch.utils.data.DataLoader(tr, batch_size=64, shuffle=True, num_workers=2)
dl_va = torch.utils.data.DataLoader(va, batch_size=64, num_workers=2)
print("Classes :", tr.classes, "|", len(tr), "images d'entraînement")

net = models.mobilenet_v3_large(weights="IMAGENET1K_V2")
net.classifier[3] = nn.Linear(net.classifier[3].in_features, len(tr.classes))
net = net.to(dev)
opt = torch.optim.AdamW(net.parameters(), lr=3e-4)
for epoch in range(8):
    net.train()
    for x, y in dl_tr:
        opt.zero_grad()
        nn.functional.cross_entropy(net(x.to(dev)), y.to(dev)).backward()
        opt.step()
    net.eval(); ok = tot = 0
    with torch.no_grad():
        for x, y in dl_va:
            ok += (net(x.to(dev)).argmax(1).cpu() == y).sum().item(); tot += len(y)
    print(f"passe {epoch + 1} : précision validation {ok / tot:.1%}")

torch.onnx.export(net.cpu().eval(), torch.randn(1, 3, 224, 224), f"{SORTIE_CLS}/classifieur.onnx",
                  input_names=["image"], output_names=["scores"], dynamic_axes={"image": {0: "n"}})
json.dump(tr.classes, open(f"{SORTIE_CLS}/classes.json", "w"))
print("Exporté dans", SORTIE_CLS)